# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [ ]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)
(234, 16) (1872, 5)


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:** **Line chart, indexed to 1970 = 100.** Change over time is what lines are for, and indexing every continent to its own 1970 size answers "relative to 1970" directly. Africa is drawn thick in purple and called out with an annotation; the other continents are thin grey lines with muted labels, and a dotted line at 100 marks "no growth". No legend, no gridlines: names are on the lines.

In [ ]:
# Q1
ACCENT, GREY = '#7B3294', '#B5B5B5'

cont = df_long.groupby(['Continent', 'Year'], as_index=False)['Population'].sum()
cont['Index'] = cont['Population'] / cont.groupby('Continent')['Population'].transform('first') * 100
end = cont[cont['Year'] == 2022].set_index('Continent')['Index']
fastest = end.idxmax()

fig = px.line(cont, x='Year', y='Index', color='Continent', template='plotly_white',
              color_discrete_map={c: ACCENT if c == fastest else GREY for c in end.index},
              title=f"{fastest} is the fastest-growing continent: {end[fastest] - 100:.0f}% larger than in 1970, "
                    f"vs {end['Europe'] - 100:.0f}% for Europe")
fig.update_traces(line_width=2)
fig.update_traces(line_width=5, selector=dict(name=fastest))

for name, val in end.items():
    hot = name == fastest
    fig.add_annotation(x=2022, y=val, text=f"<b>{name}</b>" if hot else name, showarrow=False, xanchor='left', xshift=8,
                       yshift={'Oceania': 9, 'South America': -2, 'Asia': -10}.get(name, 0),
                       font=dict(size=14 if hot else 11, color=ACCENT if hot else '#777777'))
fig.add_annotation(x=1999, y=end[fastest] * 0.55, ax=-30, ay=-50, showarrow=True, arrowcolor=ACCENT,
                   text=f"Doubled by {int(cont[(cont.Continent == fastest) & (cont.Index >= 200)]['Year'].min())}", font=dict(color=ACCENT, size=12))
fig.add_hline(y=100, line_dash='dot', line_color='#BBBBBB')
fig.update_layout(showlegend=False, title_x=0, title_font_size=18, margin=dict(r=110, t=80),
                  xaxis_title=None, yaxis_title='Population index (1970 = 100)')
fig.update_xaxes(showgrid=False, range=[1969, 2030], tickvals=[1970, 1980, 1990, 2000, 2010, 2020])
fig.update_yaxes(showgrid=False, zeroline=False)
fig.show()

---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:** **Sorted bar chart (horizontal).** Ranking 15 named countries by one quantity is a sorted-bar job; horizontal bars give long names room and sorting makes the order obvious. Colour carries the one thing the question asks about: African countries are purple, all others grey, and the title gives the count. Bars are labelled with their values, so axis, gridlines and legend are removed.

In [ ]:
# Q2
ACCENT, GREY = '#7B3294', '#C4C4C4'

g = df.assign(Gain=(df['2022 Population'] - df['1970 Population']) / 1e6)
top = g.nlargest(15, 'Gain').sort_values('Gain')              # ascending -> biggest drawn at the top
n_af = int((top['Continent'] == 'Africa').sum())
share = top.loc[top['Continent'] == 'Africa', 'Gain'].sum() / top['Gain'].sum()

fig = px.bar(top, x='Gain', y='Country', orientation='h', template='plotly_white',
             text=top['Gain'].map(lambda v: f"+{v:,.0f}M"),
             title=f"India and China added the most people since 1970; {n_af} of the top 15 are African,<br>"
                   f"but together they account for only {share:.0%} of the top-15 gain")
fig.update_traces(marker_color=[ACCENT if c == 'Africa' else GREY for c in top['Continent']],
                  textposition='outside', cliponaxis=False)
fig.update_yaxes(categoryorder='array', categoryarray=top['Country'].tolist(), title=None)
fig.update_xaxes(visible=False, range=[0, top['Gain'].max() * 1.15])
fig.add_annotation(x=0.98, y=0.15, xref='paper', yref='paper', showarrow=False, xanchor='right',
                   text='<b style="color:#7B3294">■</b> African countries   <b style="color:#C4C4C4">■</b> Other', font=dict(size=12))
fig.update_layout(title_x=0, title_font_size=18, margin=dict(t=90, r=30), bargap=0.3)
fig.show()

---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** **Scatter plot on log-log axes, with one fitted line per continent.** Area vs population is a two-variable relationship, so a scatter plot. Log scales are necessary because both axes span 5+ orders of magnitude. Colour = continent and a thin fitted line per continent shows whether the relationship differs by continent (the lines sit at different heights). The countries furthest from the overall trend are labelled directly.

In [ ]:
# Q3
PAL = {'Africa': '#D55E00', 'Asia': '#0072B2', 'Europe': '#CC79A7',
       'North America': '#E69F00', 'South America': '#56B4E9', 'Oceania': '#555555'}   # Okabe-Ito, no red/green pair

d = df.copy()
lx, ly = np.log10(d['Area']), np.log10(d['2022 Population'])
r = np.corrcoef(lx, ly)[0, 1]
slope, icpt = np.polyfit(lx, ly, 1)
d['Resid'] = ly - (icpt + slope * lx)
out = d['Resid'].abs().nlargest(7).index
d['Label'] = np.where(d.index.isin(out), d['Country'], '')

fig = px.scatter(d, x='Area', y='2022 Population', color='Continent', text='Label', hover_name='Country',
                 log_x=True, log_y=True, color_discrete_map=PAL, template='plotly_white', opacity=0.75,
                 title=f"Area and population move together (r = {r:.2f} on log scales), but each continent sits at its own level;<br>"
                       f"Greenland and the Falkland Islands (far below) and Macau (far above) break the pattern most")
fig.update_traces(marker_size=7, textposition='middle right', textfont_size=11, selector=dict(mode='markers+text'))

for cname, grp in d.groupby('Continent'):                       # one fitted line per continent
    s, i = np.polyfit(np.log10(grp['Area']), np.log10(grp['2022 Population']), 1)
    xs = np.array([np.log10(grp['Area'].min()), np.log10(grp['Area'].max())])
    fig.add_scatter(x=10 ** xs, y=10 ** (i + s * xs), mode='lines', line=dict(color=PAL[cname], width=1.5),
                    hoverinfo='skip', showlegend=False)

fig.update_layout(title_x=0, title_font_size=17, margin=dict(t=100), legend_title_text='',
                  xaxis_title='Land area, km² (log)', yaxis_title='2022 population (log)')
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)
fig.show()

---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** **Horizontal box plot on a log axis, continents sorted by median.** Box plots show the median, quartiles and outliers of each distribution, so they show the *spread* (not only the average) within each continent while allowing comparison across continents. Horizontal layout reads like a ranking; the log axis is needed because density spans 0.03 to 23,000 people/km². Boxes are one neutral grey with Asia highlighted as the continent with the widest density range; the labelled dots are the extreme countries.

In [ ]:
# Q4
ACCENT, GREY = '#0072B2', '#9E9E9E'

med = df.groupby('Continent')['Density'].median().sort_values()
order = med.index.tolist()
rng = df.groupby('Continent')['Density'].agg(['min', 'max'])
orders = np.log10(rng['max'] / rng['min'])
widest = orders.idxmax()

fig = px.box(df, y='Continent', x='Density', color='Continent', points='outliers', log_x=True, hover_name='Country',
             category_orders={'Continent': order}, template='plotly_white',
             color_discrete_map={c: ACCENT if c == widest else GREY for c in order},
             title=f"Medians range from {med.min():.0f} to {med.max():.0f} people per km² across continents, "
                   f"but every continent spans at least {orders.min():.0f} orders of magnitude inside")
fig.update_traces(marker_size=5, line_width=1.5, boxmean=False)

for cname, idx in df.groupby('Continent')['Density'].idxmax().items():           # label densest country per continent
    fig.add_annotation(y=cname, x=np.log10(df.loc[idx, 'Density']), text=f"{df.loc[idx, 'Country']} ({df.loc[idx, 'Density']:,.0f})",
                       showarrow=False, yshift=16, font=dict(size=10, color='#555555'))

fig.update_layout(showlegend=False, title_x=0, title_font_size=17, margin=dict(t=90),
                  xaxis_title='People per km², 2022 (log scale)', yaxis_title=None)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)
fig.show()

---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** **Choropleth map with a diverging scale centred on zero.** The question asks "where", so a map; colour encodes % change 2000–2022, with orange for shrinkage, near-white for no change and blue for growth (CVD-safe, not red/green). The scale is centred on 0 and capped at ±50% so outliers don't wash out the rest, and the Robinson projection avoids the visual exaggeration of high-latitude countries. A slim colour bar replaces the legend.

In [ ]:
# Q5
d = df.copy()
d['Change'] = (d['2022 Population'] / d['2000 Population'] - 1) * 100
d['Shown'] = d['Change'].clip(-50, 50)
n_grew, n_shr = int((d['Change'] > 0).sum()), int((d['Change'] < 0).sum())
shr_eu = int(((d['Change'] < 0) & (d['Continent'] == 'Europe')).sum())

fig = px.choropleth(d, locations='CCA3', locationmode='ISO-3', color='Shown', hover_name='Country',
                    hover_data={'Change': ':.1f', 'Shown': False, 'CCA3': False},
                    color_continuous_scale=['#B35806', '#F7F7F7', '#2166AC'], color_continuous_midpoint=0,
                    range_color=(-50, 50), template='plotly_white',
                    title=f"Populations grew almost everywhere since 2000 ({n_grew} of {n_grew + n_shr} countries); "
                          f"shrinkage is concentrated<br>in Europe ({shr_eu} of the {n_shr} shrinking countries), "
                          f"above all Eastern Europe and the Baltics")
fig.update_traces(marker_line_color='#999999', marker_line_width=0.3)
fig.update_geos(showframe=False, showcoastlines=False, projection_type='robinson')
fig.update_coloraxes(colorbar=dict(title='% change', orientation='h', thickness=10, len=0.4, x=0.5, y=-0.02,
                                   tickvals=[-50, -25, 0, 25, 50], ticktext=['≤ −50%', '−25%', '0', '+25%', '≥ +50%']))
fig.update_layout(title_x=0, title_font_size=17, margin=dict(t=90, l=0, r=0, b=40))
fig.show()

---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** **Treemap.** Showing a total split across two nested levels (continent, then country) in one figure is the treemap's strength: tile area = 2022 population. Asia is the story of the question, so it is the only coloured continent (blue); the other continents are shades of grey, which keeps their borders distinguishable without a legend. Every tile shows its name and population in millions.

In [ ]:
# Q6
ASIA = '#0072B2'
GREYS = {'Africa': '#8F8F8F', 'Europe': '#B0B0B0', 'North America': '#777777',
         'South America': '#C6C6C6', 'Oceania': '#DADADA'}

d = df.copy()
d['Pop (M)'] = d['2022 Population'] / 1e6
tot = d['Pop (M)'].sum()
asia = d.loc[d['Continent'] == 'Asia', 'Pop (M)'].sum() / tot

fig = px.treemap(d, path=[px.Constant('World'), 'Continent', 'Country'], values='Pop (M)', color='Continent',
                 color_discrete_map={'Asia': ASIA, **GREYS}, template='plotly_white',
                 title=f"More than half of the world's {tot / 1e3:.1f} billion people ({asia:.0%}) live in Asia; "
                       f"Africa is a distant second")
fig.update_traces(root_color='white', marker_line_color='white', marker_line_width=1.5,
                  texttemplate='%{label}<br>%{value:,.0f}M', textfont_size=12,
                  hovertemplate='%{label}: %{value:,.0f}M (%{percentRoot:.1%})<extra></extra>')
fig.update_layout(title_x=0, title_font_size=18, margin=dict(t=80, l=10, r=10, b=10))
fig.show()

---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:** **100% stacked column chart.** The question is about each continent's *share* of a total that changes (3.7 to 8 billion), so every year is a column that sums to 100%. Africa (purple) is stacked on the bottom and Europe (orange) on the top, so both change against a flat edge and their trend is readable; Asia and the rest are grey. Only Africa and Europe carry value labels, which keeps the focus on the two stories, and series names are placed directly at the right edge.

In [ ]:
# Q7
d = df_long.copy()
d['Group'] = np.where(d['Continent'].isin(['Africa', 'Asia', 'Europe']), d['Continent'], 'Rest of the world')
sh = d.groupby(['Year', 'Group'], as_index=False)['Population'].sum()
sh['Share'] = sh['Population'] / sh.groupby('Year')['Population'].transform('sum') * 100
sh['Year'] = sh['Year'].astype(str)
p = sh.pivot(index='Year', columns='Group', values='Share')

order = ['Africa', 'Rest of the world', 'Asia', 'Europe']          # bottom -> top
cols = {'Africa': '#7B3294', 'Europe': '#E69F00', 'Asia': '#8E8E8E', 'Rest of the world': '#D0D0D0'}

fig = px.bar(sh, x='Year', y='Share', color='Group', template='plotly_white',
             category_orders={'Group': order, 'Year': sorted(sh['Year'].unique())}, color_discrete_map=cols,
             title=f"Africa's share of humanity grew from {p.loc['1970', 'Africa']:.1f}% to {p.loc['2022', 'Africa']:.1f}%, "
                   f"roughly offsetting Europe's fall from {p.loc['1970', 'Europe']:.1f}% to {p.loc['2022', 'Europe']:.1f}%")
fig.update_traces(marker_line_color='white', marker_line_width=1)
for g in ('Africa', 'Europe'):
    fig.update_traces(text=sh[sh['Group'] == g].sort_values('Year')['Share'].map(lambda v: f"{v:.0f}%"),
                      textposition='inside', textfont=dict(color='white', size=12), selector=dict(name=g))

base = 0
for g in order:                                                   # direct labels at the right edge, at 2022 mid-heights
    h = p.loc['2022', g]
    fig.add_annotation(x=7, y=base + h / 2, text=f"<b>{g}</b>", showarrow=False, xanchor='left', xshift=30,
                       font=dict(size=12, color=cols[g] if g in ('Africa', 'Europe') else '#666666'))
    base += h

fig.update_layout(showlegend=False, title_x=0, title_font_size=17, margin=dict(t=90, r=150),
                  xaxis_title=None, yaxis_title=None, bargap=0.15)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(visible=False, range=[0, 100])
fig.show()